# Megatron-style tensor-parallel MLP

Tensor parallelism splits one layer's weights across devices. For a transformer MLP $y = \mathrm{gelu}(x W_1 + b_1) W_2 + b_2$, the standard split needs only **one** all-reduce:

- $W_1$ `(D, F)` and $b_1$ `(F,)` are split by **columns**: device $n$ gets columns $[nF/N, (n+1)F/N)$. GELU is elementwise, so each device can apply it to its own slice of the hidden layer.
- $W_2$ `(F, D)` is split by **rows**, with the matching row range. Each device computes a partial product $h_n W_2^{(n)}$ of shape `(B, D)`.
- The partial products are summed with `psum`; then $b_2$ is added once.

Write `tp_mlp(x, w1, b1, w2, b2, n_shards)`:

- `x`: `(B, D)`; full weights as above; `n_shards` (a Python int) divides `F`. GELU is the exact (erf) form, `jax.nn.gelu(z, approximate=False)`.
- Split the weights yourself, then run a per-device function with `jax.vmap(..., axis_name='tp')` (one CPU device simulates $N$).

Return `(y, partials)`: `y` is `(B, D)`, the output from device 0 (all devices hold the same result after the all-reduce); `partials` is `(N, B, D)`, each device's $h_n W_2^{(n)}$ **before** the all-reduce. float32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def tp_mlp(x, w1, b1, w2, b2, n_shards):
    """Return (y (B, D), partial products before the all-reduce (N, B, D))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 2.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.5, 0.5], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[1.3413447141647339, 2.4544997215270996]], "dtype": "float32"}, {"__nd__": [[[0.8413447141647339, 0.0]], [[0.0, 1.9544997215270996]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, -1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[0.8413447141647339, -0.15865527093410492]], "dtype": "float32"}, {"__nd__": [[[0.8413447141647339, -0.15865527093410492]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.1381776648333883, 0.7712193621153433, 1.150088000505753, 0.7354942141085169], [1.4479647542823046, 0.0033085746743845325, -1.4230297964925127, -0.057385155364924105], [0.48261705395315346, 0.4151884302474164, -0.600791259405733, -2.0598127253426366]], "dtype": "float64"}, {"__nd__": [[1.047450559743749, -0.0013777611604841617, 0.232826403100379, -0.3188506654115649, -0.41193794457878447, 0.3218459590507071, 0.4757547954936833, -0.47132672923157376], [-0.28117292980638064, -0.35116305925814817, -0.5450061325164669, -0.41608207001034375, -0.06001615025865583, -0.8732390359309729, 1.072073473054031, -0.716934873231709], [-0.19689736011848108, 0.8906804891512248, -0.21093096787802693, 0.021934585543133496, 0.3118306428777072, 0.8491024567456408, -0.4501546272823329, 0.7590012477798019], [-0.5564831558777434, 0.9486499141089901, 0.5711935285018864, 0.2768755315511087, -0.15347880690840301, 0.7968768654423743, 0.4657475617381182, 0.641544301190047]], "dtype": "float64"}, {"__nd__": [-0.12947386107117784, 0.0023891640537416904, 0.034306086392037534, 0.06132177977896639, -0.16333879515796132, -0.1606089471945136, 0.0534518590189143, -0.03824199228846602], "dtype": "float64"}, {"__nd__": [[0.12315521916808872, -0.5536103924809034, 0.06524503074023118, -0.44303419604357425], [-0.5908375842022465, -0.5843385116223273, 0.17205485319535535, -0.8313905530938899], [-0.9045446182494192, -0.44509919876266624, 0.14031097970935352, 0.3143645396001066], [0.3292202659582701, 0.46745850353623863, -0.280934028917245, -0.2375790684645592], [0.7769503856147769, -0.07665835207838066, 0.14710137292385625, -0.4767430665874835], [0.2237842691575521, -0.6356985463132175, -0.24633480181930956, -0.14524323668542757], [-0.12740480298758972, 0.28001583074200076, 0.36640575416867127, 0.43176648909930887], [-0.004198921422229737, -0.3400416392959008, -0.07673453062008212, -0.25757181940719]], "dtype": "float64"}, {"__nd__": [-0.9207692839678769, 1.5873834859253573, -1.2792001601119882, 0.6419841286690195], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[-1.2095413208007812, 0.43755269050598145, -1.1902047395706177, -1.0228301286697388], [-1.4237977266311646, 0.9653258323669434, -0.6082028746604919, 0.8493653535842896], [-0.7473458051681519, 0.8211638331413269, -1.1636484861373901, 0.07888948917388916]], "dtype": "float32"}, {"__nd__": [[[-0.5960918664932251, -0.605707049369812, 0.1501123309135437, -1.2045336961746216], [-0.2010221928358078, -1.1041375398635864, 0.19185979664325714, -0.43394023180007935], [0.2591070532798767, -0.7968430519104004, 0.11788111925125122, -0.6288064122200012]], [[0.3073198199272156, -0.5441237688064575, -0.06111682578921318, -0.46028056740760803], [-0.3020062744617462, 0.4820798933506012, 0.47913751006126404, 0.6413214802742004], [-0.08568362146615982, 0.03062335215508938, -0.0023293776903301477, 0.06571178883314133]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.5618076311846935, -1.8733147562700287, 1.964610287499639, -0.4975203263832242], [-1.2158330285202374, 2.234285565108449, -0.4009591391394682, 0.023373544841014615], [1.4974306555934753, -1.0990343326886598, -1.0793869703519559, 0.6312855007908246]], "dtype": "float64"}, {"__nd__": [[0.17356022745154714, -0.4708041067095833, -0.799547079327153, -0.7420571787203999, 0.22778454955667277, 0.6231467857527739, 0.5948255722947011, 0.20634417934867164], [0.19045744102867213, -1.1392791805457738, -0.02543409693542538, -0.2750467352505998, 0.6864823543477888, -0.11528540331048608, 0.05764815095146123, 0.08452926321289116], [0.21796804965520636, 0.2785788191870292, 1.09611466630614, 0.7585215810299062, 0.2215664799299455, -0.3709048143358578, 0.8945675914760595, 0.4569486641851617], [0.06148771274326352, -0.26211347062857804, -0.49503396230716373, 0.544078704029065, 0.5608174650606316, 0.04080933824626452, 0.01446188585240037, 0.937457636926531]], "dtype": "float64"}, {"__nd__": [0.057521019810402745, -0.012008989409356914, -0.06688005756207548, 0.10105856570497125, 0.08817644749704166, -0.10563825107738799, -0.22238325367702233, 0.07095975137101196], "dtype": "float64"}, {"__nd__": [[0.3554127678337745, 0.561619322125511, 0.37787990124170806, -0.8813887139107646], [0.583495049564484, -0.17958163288866394, 0.778386886029576, 0.1805444825867734], [-0.15736510270339354, -0.5594337067602327, -0.24653318883863382, 0.2045296515315839], [-0.06640880997486713, 0.6448563175410065, -0.0019415045766995225, 0.18703950746868242], [-0.335373742361834, -0.33840371659229934, -0.2859222439993679, -0.4801834940812093], [0.54033061571595, 0.7782515575621232, -0.29151402027158535, -0.17665728892293794], [0.6585449099526575, 0.31559659344340507, 0.4119196285725056, -0.5254610347353912], [0.9980909365805182, 0.2223200473413169, 0.20357737174757962, -0.11518529022079198]], "dtype": "float64"}, {"__nd__": [-0.06400461468790107, -0.027193925221346393, -0.21342599868804374, 0.35085491137694924], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[1.8415746688842773, -0.4722164571285248, 1.968701958656311, 1.486151933670044], [-0.7210585474967957, -0.6424797177314758, -0.6169470548629761, -0.12399342656135559], [0.8323075175285339, 0.887128472328186, -0.5433295369148254, 0.22242224216461182]], "dtype": "float32"}, {"__nd__": [[[1.7862803936004639, -0.5494735836982727, 2.382870674133301, 0.552283763885498], [0.016183890402317047, 0.06666430085897446, 0.011313219554722309, -0.10084296762943268], [0.006621605716645718, -0.029595432803034782, 0.012785903178155422, 0.04276306927204132]], [[-0.5920401811599731, -0.14429260790348053, -0.7003067135810852, 0.9936375021934509], [-0.04440280795097351, -0.1116601899266243, -0.06420943140983582, 0.06299258023500443], [0.010655062273144722, -0.08550616353750229, 0.002392842900007963, -0.027956420555710793]], [[-0.03872694447636604, -0.07626459747552872, 0.08701293915510178, 0.09619680047035217], [-0.46916407346725464, -0.5109109878540039, -0.2789211571216583, -0.5187904834747314], [0.7127483487129211, 1.0139122009277344, -0.34361860156059265, -0.1813339740037918]], [[0.7500659823417664, 0.3250082731246948, 0.41255101561546326, -0.5068210959434509], [-0.15967091917991638, -0.05937891826033592, -0.0717037171125412, 0.08179251849651337], [0.1662871092557907, 0.015511751174926758, -0.0014636724954470992, 0.038094669580459595]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.9161097009883893, -0.5259581391082356, -0.10989380293161596, 0.8601942093539285, 0.5859257079791832, 3.275741684642875], [-0.8835171809034402, 0.7699071007019623, -0.22443548892264845, -0.06886276556621189, -0.029069679001691814, -0.04875037473936539]], "dtype": "float64"}, {"__nd__": [[0.4135199791934579, 0.18626348101523119, 0.5270463595149785, 1.4953930200886756, 0.533220509452383, -0.007313968592782607, 0.6954759290250604, -1.4005240236864733, 0.7335287472618938, 0.9773137985697778, 0.07864875387784867, 0.2400236662781427], [-0.04434795269885516, -0.4630964909363858, 0.3772577147332565, 0.27761509043094085, -0.18128643602996364, 0.6794140533833377, -0.46943256001941636, 0.5521307662820325, 0.4292117005634304, 0.32593610999706285, 0.2903001518708959, 0.23419874881877792], [-0.6012797268951054, -1.0762223171242584, 0.03839887369439064, 0.8129797610864834, 0.14801115656208214, 0.2691385620456972, 0.2550895249441641, 0.24804381610904885, 0.22278992330326802, 0.36943600923014963, -0.9621944919674791, 0.10105484630765538], [-0.7002093271776811, 0.5646052387022051, -0.7489675933964155, -0.3505238474911646, 0.3544924417807528, -0.14147769785262368, 0.2635766496028424, -0.505440442975328, 1.3110065900903434, -0.03359847933895234, -0.5989614229942978, 0.4908851083947186], [-0.29367905078087875, 0.06568808181736763, -0.023066989644049112, 0.17340901361575412, 0.023695365424997564, 0.8533304539989243, 0.5610972922569877, -0.3841014412963506, 0.4196662981842278, -0.10287079843534414, -0.06606749901333282, -0.20472727455134468], [-0.5993377155562848, 0.3259193200162939, -0.348830114026222, 0.11868319559329785, 0.649399435478272, -0.5298639375326707, -0.09285021988142017, 0.06474290632535801, -0.1925938082424429, -0.1567278045706997, -0.40236913237062805, -0.9571634265809914]], "dtype": "float64"}, {"__nd__": [0.006637231950663228, 0.2114761831337464, -0.08677082832703042, 0.043273664403007533, 0.01935336607529328, -0.06286277054039334, -0.02860177499364241, 0.08938109996050278, 0.0006591316298625247, -0.006859812598535399, 0.08331163565845862, -0.03495532857254375], "dtype": "float64"}, {"__nd__": [[-0.527121170706896, -0.11350728946040371, -0.2358427310614509, -0.009191197135463144, 1.2313570730339114, -0.3982790839154406], [-0.37852087986222566, 0.4445052350767855, 0.5897888799434423, -0.25561306183669263, -0.08127216649302474, -0.42566894631632995], [0.48449988115943193, -0.008690555008288017, 0.17780243455926512, -0.43080043582199384, 0.15106641950827018, -0.05095040507679652], [0.14290525381395194, 0.15293395187766096, -0.9631796034147282, -0.6388991069546824, 0.9930864121924902, 0.05631179239785496], [0.4406143228155253, 0.07597539297336041, -0.005359487401680285, 0.39170261842415194, -0.0890460828024874, 0.10318072024293903], [0.34198197408867864, -0.43858926051112573, 0.5453086086854626, -0.373761263818909, -0.9449478036728157, -0.5982332809264341], [-0.044908260791497866, 0.9686356375939064, -0.1706250666905326, -0.04899441810836391, -0.3338512979847732, 0.3846242703233484], [0.39770686477745454, 0.4507548933979972, 0.23551641002754367, -0.6244707194626922, 0.06459765828921939, -0.3992835042175702], [0.04918166565770921, -0.29114933255848563, -0.33320230345140206, 0.14837793926813003, -0.1605364645883655, 0.22646239827841091], [0.2972212303131286, -0.5743040457034284, 0.33543498354170287, 1.3205818933362221, 0.4875107180609879, -0.3176264190769458], [0.5836323042081624, 0.6421463138396509, 0.038869084502011604, -0.3759008674208095, -0.445370829759067, -0.05787055071576773], [-0.23572003506760025, -0.22367864250217573, 0.12872992668081432, 0.5312602088927286, 0.02611394020451737, -0.37454800632602997]], "dtype": "float64"}, {"__nd__": [0.09067485221448353, -1.3620758437229883, -1.5335947361381417, -0.03844384837868619, -0.034944352159618684, 0.2672849800028427], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": [[0.6414076089859009, 0.6322243213653564, -1.9338207244873047, 0.030428633093833923, 0.33416974544525146, 0.32085415720939636], [0.9411221742630005, -0.5471286177635193, -0.8629087209701538, -1.5028800964355469, -0.5753101706504822, -0.5768411755561829]], "dtype": "float32"}, {"__nd__": [[[-0.7249203324317932, 1.2229875326156616, 0.1516520231962204, -1.3518052101135254, 1.0067046880722046, -0.8991208076477051], [0.00024440535344183445, -0.036322932690382004, 0.09848041832447052, 0.13716568052768707, -0.23278789222240448, 0.050243135541677475]], [[1.2519874572753906, 1.1340004205703735, -0.2204892486333847, 1.1954385042190552, -0.5227821469306946, 0.7266536355018616], [0.6899569630622864, 0.47246912121772766, 0.5697581171989441, -1.2033408880233765, -0.08788784593343735, -0.9061101675033569]], [[0.02366558276116848, -0.362687885761261, -0.3313887119293213, 0.22523918747901917, -0.11480845510959625, 0.22603633999824524], [0.16024596989154816, 0.37880098819732666, 0.002447474980726838, -0.39826101064682007, -0.21969011425971985, 0.011740860529243946]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.2907316380790337, 0.20443642744729296, -1.5936272708339951, -0.08222157021355363, -1.7017282599107113, 0.10365955953089649, 0.028585457626774433, -0.6880786119996951], [0.5759272717100151, 0.8240232838685334, 1.0872669191532915, 0.513569847555992, 0.9133864774156525, -1.2652941444849337, 1.4810081775094328, 0.3456308818008566], [-1.0005134585192847, 0.040344921834207084, -0.8221436960263045, 0.49813947042289247, 0.39750187983998136, 0.2145122676877448, -1.191190444490404, -0.2431683942171218], [1.4774912865557663, 0.9656027342646929, -0.5105481553278916, -1.3228556644133596, -0.31599024795071934, 0.7866908861066657, 0.25711274486973296, 1.1182836308108972], [0.6114695737140202, -1.1579840215563242, 0.250073855698667, -1.4802810778777014, 0.02205724760558896, 0.49133376104809384, 0.683959822839259, 0.6319447795558012]], "dtype": "float64"}, {"__nd__": [[0.20442265695300543, 0.1216217227988268, 0.17059123680644878, -0.3825534927002176, 0.11270746271126016, 0.0032044290363941502, -0.0031883165229840446, 0.4638376715183016, 0.2082005549458945, -0.11635936995360288, 0.24742135674982418, -0.4093086624927898, 0.1213407616415153, -0.4679164356842771, 0.28264315927534933, 0.53050113728105], [0.013373818254870743, -0.20695154779014296, -0.5214670052976368, 0.1899342191801146, -0.08854189650781193, -0.04163318556446466, 0.1479524373883988, -0.2612039714927603, 0.1368260401343836, 0.5657934891901222, -0.18449761387731378, -0.04405178516210628, 0.17936147149482326, 0.1184704973998513, -0.26508482312128895, -0.1559165459022078], [-0.3677245184433692, 0.11811810006134692, -0.6965413913631529, -0.08991868811685297, -0.4322116335129264, -0.31871652033210884, -0.5713328481267645, -0.02220087861556756, 0.3171275598061751, 0.3287057947348422, -0.4508180216658013, 0.23949872718691978, 0.15583265263586174, 0.09931915666073843, 0.227925742084024, 0.17428217311506403], [-0.9024630219504306, 0.30761958708488446, 0.12137442179874854, -0.2287363614733215, -0.45255428794975544, -0.17343467408970906, 0.14461093113882564, 0.22097784843750964, -0.14900212838984847, 0.0970340922583756, -0.043053798625682614, 0.23443609516058772, -0.3280772766312646, 0.31116473282633367, 0.4546288221965424, 0.5577800402619901], [0.4414301485166416, 0.41448805251578313, 0.0326375473544866, -0.08476598523098988, 0.11422389749310893, 0.08749612322729526, 0.06426536304612435, 0.044953801129006356, -0.09074200345185371, 0.27576619946230657, -0.021045595716562733, 0.4256627027839845, 0.0014220464064231848, -0.24646426422349632, 0.20215452733090103, 0.25373954241173513], [-0.39609421370225606, -0.11783463002834481, -0.27678693894220385, 0.16395922752423278, 0.2597193901430642, -0.28054706235890925, -0.3279117860812228, -0.2647818646913407, 0.009805895607166397, 0.06122627149199283, 0.05004131477826623, -0.11827173296285208, 0.0011316922761395792, 0.23288063819750843, -0.42095145034625114, -1.978652607795641e-05], [-0.5720095441408954, -0.2197474662680659, 0.12435771832030718, -0.200902763035193, -0.14992040643915858, 0.1636359712603839, 0.0737090969827451, 0.05615351441725291, 0.09990947589903679, 0.2268351237992829, 0.5196986447603116, -0.18185936850950593, 0.5319520813317323, -0.3834428209631084, -0.07572468211577457, 0.11507704262641488], [0.018821637852349474, 0.16931840463107486, -0.4062713567906506, 0.10271465432383871, -0.23415558033222772, 0.4166142817438037, -0.20533400094784648, 0.11410635172757513, -0.0697877360302088, 0.37659300532330636, -0.26970291969295734, -0.1687702945134692, 0.3213999110290823, 0.16671428567724125, 0.2123320636994008, -0.36849364701996745]], "dtype": "float64"}, {"__nd__": [0.028442356002331273, -0.0866749708741707, -0.11863573508767243, -0.09081536129071938, 0.027006146833229185, 0.115227943150033, -0.0392161311040089, 0.17886025683671206, 0.1234492704135452, 0.005817492849275925, -0.01095149328940333, 0.02049035940902041, -0.07826915727552469, -0.1472999817111301, -0.12162354475635669, 0.01763208194301587], "dtype": "float64"}, {"__nd__": [[-0.08186003000932998, -0.5814501627964201, -0.5223832592075798, 0.14810608215721838, 0.09325758601854334, -0.0522969978513375, 0.07344320175654478, -0.2187774873383087], [-0.06861300112981811, -0.04362331597068203, 0.34571849266563676, -0.08645411980530346, 0.2099493689941054, 0.04812660561425367, -0.3100221911190063, 0.3983734831642749], [-0.048037251020803244, -0.2385737095198274, -0.12983171657225828, 0.044236509021127544, -0.215628072796967, 0.45993206231318856, -0.41242623616728563, 0.34464732660696573], [-0.7514023630002785, -0.1796408903317034, 0.18998030878313682, -0.027086545016748977, -0.3742478891106139, 0.20499465481558982, 0.036874358897696254, 0.184547479008654], [-0.460031256713703, 0.11080801201692984, 0.10763851051423817, 0.47117494352694433, 0.328165984821936, 0.47207342756489235, -0.15128187098181187, 0.28042967745712677], [0.4682688322934721, -0.2615031987933585, 0.1136139464420146, -0.3210963522539349, -0.13167867721386362, 0.34421529235709686, 0.029497678763588366, 0.19341839323599685], [0.28636625550741784, -0.32737305740660644, -0.4371791180535686, -0.27421127574463183, -0.23918605464866505, 0.5053479521577663, 0.33210398301015753, 0.122550036837572], [0.3041923446305586, -0.0009143528150569991, -0.5088831651140601, -0.47249359249382067, 0.3211739138281488, -0.19732342586946613, -0.2698013683594423, 0.28928477781797085], [0.010860848760476938, 0.06401608769632744, 0.04856358491194068, -0.5719479276859506, 0.29082082801047276, -0.02337263098658528, 0.1810892130965572, -0.26324913063962574], [-0.11940471597144592, -0.3079504851951361, -0.2750099156103569, -0.1385993551307737, -0.20847396818213657, 0.009271906038663591, 0.09284044482222516, -0.05067262349350716], [0.4414403805194628, -0.3642503418141819, 0.33154290139443054, 0.035598273922250616, -0.2516903389245553, 0.3336446217339334, 0.3649174445173056, 0.18960033404044754], [0.07684932691725628, 0.5735051570506559, 0.22372153327945662, 0.18089784743998838, 0.019924708084298425, -0.2706499386076075, -0.4938320563237619, 0.41315053821730835], [-0.20211407166336562, 0.13614983330019095, -0.04221378844616471, 0.41981653243209577, 0.18279446678352804, -0.1468861713541023, 0.27270554329602775, 0.18602943097670027], [0.23504424017999365, 0.1811372588725481, -0.14718868960045983, -0.3855281501405212, -0.1195298907855159, 0.13696117984611503, -0.07479905403742663, -0.12096733597417053], [-0.07699824116912761, 0.4698613142889954, 0.46079358932609565, -0.35547586186807084, 0.10139512048800228, -0.05607545485424261, -0.23767839739842228, -0.0024889989860452733], [-0.21015692467697322, -0.18912449452683555, 0.265550366639725, 0.19069979831424114, 0.1166154838591632, -0.037909347899717, -0.22718807346212727, -0.01752525777022367]], "dtype": "float64"}, {"__nd__": [-1.6024209936403429, -0.09242151512252243, -0.8074410419577903, 0.8883954822078672, -1.1675821945688933, -0.36370061092131956, 0.3507043841306818, -0.6932570418575451], "dtype": "float64"}, 8], "e": {"__tup__": [{"__nd__": [[-1.6238619089126587, -0.6484842896461487, -0.9430798292160034, 0.7967373728752136, -1.8805713653564453, 0.9725384712219238, 0.5106620788574219, -0.22295239567756653], [-1.6228691339492798, 0.01756209135055542, -0.685337483882904, 0.22254568338394165, -0.7094485759735107, -0.7995852828025818, 0.19086861610412598, -0.37356331944465637], [-1.7637591361999512, 0.04316131770610809, -0.9893448948860168, 0.9569312334060669, -1.3851739168167114, -0.05957278609275818, -0.02342018485069275, -0.3759393095970154], [-1.855041742324829, -0.8441542387008667, -1.4235117435455322, 1.3199976682662964, -0.7342007756233215, -0.14784391224384308, 0.9868575930595398, -0.6372722387313843], [-1.541793704032898, -0.5798161625862122, -1.1412547826766968, 1.2213821411132812, -0.727831244468689, -0.16479399800300598, 0.7769521474838257, -0.5512765645980835]], "dtype": "float32"}, {"__nd__": [[[0.019472653046250343, 0.08471758663654327, 0.029620172455906868, -0.009794662706553936, -0.03810129314661026, 0.0013328249333426356, 0.027410808950662613, -0.01822402887046337], [-0.0021343184635043144, 0.08716981112957001, 0.16707919538021088, -0.04470350965857506, 0.032822154462337494, 0.019843606278300285, -0.08377568423748016, 0.12845319509506226], [-0.03236697241663933, -0.1786433458328247, -0.11603223532438278, 0.03423328697681427, 0.0520966500043869, -0.010054941289126873, -0.0138847129419446, -0.01930183917284012], [-0.07062406837940216, -0.5763026475906372, -0.5825225114822388, 0.16321104764938354, 0.058283764868974686, -0.06059182807803154, 0.125883549451828, -0.2866315543651581], [-0.04540620371699333, -0.372244656085968, -0.3775635063648224, 0.1057509183883667, 0.036960430443286896, -0.039313364773988724, 0.08237728476524353, -0.18654298782348633]], [[-0.4233916997909546, -0.256916880607605, 0.009706850163638592, 0.016253847628831863, -0.3423101603984833, 0.4218573272228241, -0.2636026442050934, 0.33219173550605774], [0.13189610838890076, 0.07008519768714905, -0.00924482848495245, -0.003049284452572465, 0.09823767840862274, -0.11183979362249374, 0.06394387781620026, -0.08890101313591003], [-0.25031763315200806, -0.09025119990110397, 0.044278472661972046, -0.002868417650461197, -0.1503431797027588, 0.12811973690986633, -0.04325446859002113, 0.10604676604270935], [-0.030212203040719032, 0.031121468171477318, 0.03161178529262543, -0.008850893937051296, 0.017321620136499405, -0.06720511615276337, 0.07151959091424942, -0.04878193885087967], [0.07046002149581909, 0.024479646235704422, -0.013041606172919273, 0.0009945466881617904, 0.0415385439991951, -0.034244418144226074, 0.010486780665814877, -0.028495268896222115]], [[-0.14771457016468048, 0.021722856909036636, 0.05535881221294403, 0.16606363654136658, 0.12422976642847061, 0.22843267023563385, -0.06017668917775154, 0.1346672773361206], [0.21214626729488373, -0.09848341345787048, 0.021473387256264687, -0.166777566075325, -0.08685687929391861, 0.045086994767189026, 0.030097469687461853, 0.023259788751602173], [-0.12038258463144302, 0.0315721295773983, 0.02430199831724167, 0.1205533891916275, 0.08237090706825256, 0.10925023257732391, -0.03743189200758934, 0.06509034335613251], [-0.024802036583423615, -0.07754842191934586, 0.13115066289901733, 0.11443210393190384, 0.13134883344173431, 0.48866191506385803, -0.07807689905166626, 0.2840728759765625], [-0.021480828523635864, -0.057078104466199875, 0.0984518826007843, 0.08835774660110474, 0.10003530979156494, 0.3672899901866913, -0.059178318828344345, 0.21355502307415009]], [[0.15763798356056213, -0.23810777068138123, -0.23323041200637817, -0.12069766223430634, -0.2276909202337265, 0.40072816610336304, 0.28674066066741943, 0.040901876986026764], [0.2044387012720108, -0.00835398118942976, -0.3410126268863678, -0.3135053515434265, 0.20301517844200134, -0.11623360961675644, -0.16743308305740356, 0.19086940586566925], [0.046881768852472305, -0.09344366937875748, -0.06646021455526352, -0.0240720734000206, -0.1052488461136818, 0.16707681119441986, 0.12589842081069946, 0.0017830925062298775], [0.013619538396596909, 0.036465346813201904, -0.027466902509331703, -0.04022836685180664, 0.07492754608392715, -0.08610526472330093, -0.0776091068983078, 0.029698755592107773], [0.03463267907500267, 0.055174969136714935, -0.06502782553434372, -0.0826757550239563, 0.12824945151805878, -0.13947123289108276, -0.12994401156902313, 0.058293815702199936]], [[0.017671331763267517, 0.040583398193120956, 0.037435296922922134, 0.10155501216650009, -0.006837435066699982, 0.0017512724734842777, -0.040017012506723404, 0.044635310769081116], [-0.15551024675369263, -0.3835984766483307, -0.3467411696910858, -0.46413618326187134, -0.13960590958595276, 0.0011442851973697543, 0.21287570893764496, -0.1959603875875473], [0.012443742714822292, 0.026326093822717667, 0.0248882956802845, 0.10807103663682938, -0.02181692235171795, 0.002642197534441948, -0.04003259539604187, 0.0481831394135952], [-0.039291683584451675, -0.08623963594436646, -0.08062248677015305, -0.29068610072135925, 0.04537723585963249, -0.006394496653228998, 0.11001358926296234, -0.12897571921348572], [0.018266988918185234, 0.05711134150624275, 0.04861244559288025, -0.1411467343568802, 0.10739722102880478, -0.007675576955080032, 0.038436733186244965, -0.06664101034402847]], [[0.17862391471862793, -0.24450218677520752, 0.10884407162666321, -0.012232622131705284, -0.11156242340803146, 0.18512842059135437, 0.23264187574386597, 0.018758179619908333], [0.030702024698257446, 0.11177506297826767, 0.05879424512386322, 0.04008351266384125, -0.0037835233379155397, -0.04756130278110504, -0.09460249543190002, 0.09501981735229492], [-0.021533479914069176, 0.38695114850997925, 0.0800500437617302, 0.09952715784311295, 0.04922429844737053, -0.20682252943515778, -0.3408692181110382, 0.21109801530838013], [0.08807571232318878, -0.13470089435577393, 0.04998280853033066, -0.009910672903060913, -0.05642438679933548, 0.09858208894729614, 0.1270863562822342, 0.0008086232119239867], [0.14499109983444214, -0.22789037227630615, 0.0806807205080986, -0.01800040900707245, -0.09350129961967468, 0.16545812785625458, 0.2145877331495285, -0.002336135134100914]], [[0.1392383575439453, 0.05872733145952225, -0.059106387197971344, -0.24307821691036224, -0.08412327617406845, 0.08597953617572784, -0.07898810505867004, -0.08530744910240173], [-0.21417495608329773, 0.09296867251396179, -0.015068047679960728, 0.4293486773967743, 0.1796398162841797, -0.15053443610668182, 0.25235316157341003, 0.1827165186405182], [0.1684114933013916, 0.08930637687444687, -0.08205659687519073, -0.28847983479499817, -0.0967397689819336, 0.10217130929231644, -0.08249197155237198, -0.0980469137430191], [-0.23740895092487335, 0.1027374118566513, -0.01651957258582115, 0.47582927346229553, 0.19904063642024994, -0.1668330579996109, 0.2795027792453766, 0.20244891941547394], [-0.18039600551128387, 0.06931149959564209, -0.007490993477404118, 0.3589124381542206, 0.14884230494499207, -0.12589558959007263, 0.2061319798231125, 0.15137214958667755]], [[0.03702112287282944, -0.02228715270757675, -0.08426716923713684, 0.010272537358105183, -0.026593485847115517, 0.01102876104414463, 0.05594882741570473, 0.002681731479242444], [-0.22781167924404144, 0.2384207397699356, 0.5868234038352966, -0.14311015605926514, 0.17466506361961365, -0.07579042762517929, -0.3732947111129761, -0.015763605013489723], [0.0355256013572216, -0.03623471036553383, -0.0908735916018486, 0.021571246907114983, -0.027134902775287628, 0.011745009571313858, 0.05794185772538185, 0.0024651153944432735], [0.04802301898598671, -0.04726536571979523, -0.12168458104133606, 0.027805766090750694, -0.03649383410811424, 0.015742456540465355, 0.0778333991765976, 0.0033448252361267805], [0.03955959528684616, -0.036259014159440994, -0.09843479841947556, 0.02079387940466404, -0.02977103553712368, 0.01275863777846098, 0.06334961950778961, 0.002774863038212061]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0614365772546654, -2.0846012886096887]], "dtype": "float64"}, {"__nd__": [[-0.4423139184724452, 0.4708809597753604, -0.30534526006482043, -0.7330689671688414, 0.4563640620871965, 0.11388791615181355], [-0.29080391753010726, -0.06766568690835928, 0.6460099543514065, 0.9698083448911285, 0.4289557950347406, -0.14569563007063033]], "dtype": "float64"}, {"__nd__": [0.0829667198995696, 0.018433644903700995, -0.14082374811922024, -0.013842487547627833, -0.0423640658668954, 0.09404490709371707], "dtype": "float64"}, {"__nd__": [[0.46255877279825575, -0.06302563774552561], [0.2347117441104678, -0.18291240646022092], [-0.583668973367885, -0.32068353619363815], [0.5679606262158062, -0.2999776841804108], [-0.8428695889685863, -0.15164249743881295], [0.3217900937882247, -0.7971377488760255]], "dtype": "float64"}, {"__nd__": [-1.0609643671126585, 0.8603759971288719], "dtype": "float64"}, 6], "e": {"__tup__": [{"__nd__": [[-0.5464115142822266, 0.6660283207893372]], "dtype": "float32"}, {"__nd__": [[[0.22844119369983673, -0.031126102432608604]], [[0.02541675977408886, -0.019807446748018265]], [[0.058024853467941284, 0.03188042715191841]], [[-0.022141890600323677, 0.011694601736962795]], [[0.1392168402671814, 0.025046804919838905]], [[0.08559508621692657, -0.21203595399856567]]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(tp_mlp, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")